In [2]:
import textwrap
import chromadb
import numpy as np
import pandas as pd

from IPython.display import Markdown
from chromadb import Documents, EmbeddingFunction, Embeddings

from google import genai
from langchain_chroma import Chroma

In [3]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path("../.env/.env")
load_dotenv(dotenv_path=env_path)
chroma_api_key = os.getenv("CHROMA_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

python-dotenv could not parse statement starting at line 9


In [4]:
from google import genai

client = genai.Client(api_key=google_api_key)

In [5]:
for m in client.models.list():
  if 'embedContent' in m.supported_actions:
    print(m.name)

models/gemini-embedding-001


### Data ###

In [6]:
records = [
    "Product: สมุดโน้ตกระดาษคราฟท์ปกแข็งลายวินเทจ\n""Description: สมุดปกแข็ง กระดาษคราฟท์สีน้ำตาลธรรมชาติ 160 แผ่น หนา 100 แกรม พร้อมริบบิ้นคั่นหน้า\n""Suitable for: ของขวัญวันเกิด, ของที่ระลึกงานแต่ง, ของแจกพนักงาน, คนชอบความวินเทจ\n""Style: วินเทจ, เรโทร, อบอุ่น",
    "Product: ปากกาหมึกซึมลายไม้\n""Description: ปากกาหมึกซึมด้ามไม้แท้ เขียนลื่น\n""Suitable for: ของขวัญผู้ใหญ่, งานเกษียณ\n""Style: เรียบหรู",
    "Product: เทียนหอมไขถั่วเหลืองกลิ่นลาเวนเดอร์\nDescription: เทียนหอมธรรมชาติ ไร้เขม่าดำ ช่วยผ่อนคลายและหลับสบาย\nSuitable for: ของขวัญวันเกิด, ของขวัญขึ้นบ้านใหม่, คนชอบแต่งบ้าน\nStyle: มินิมอล, อบอุ่น",
    "Product: โคมไฟตั้งโต๊ะฐานไม้\nDescription: โคมไฟดีไซน์นอร์ดิก ฐานทำจากไม้โอ๊คแท้ ปรับความสว่างได้ 3 ระดับ\nSuitable for: อ่านหนังสือ, ตกแต่งห้องนอน, ของขวัญรับปริญญา\nStyle: นอร์ดิก, มินิมอล",
    "Product: ชุดถ้วยกาแฟเซรามิกทำมือ\nDescription: ถ้วยกาแฟงานปั้นมือ เคลือบสีเอิร์ธโทน เอกลักษณ์ไม่ซ้ำกันในแต่ละใบ\nSuitable for: คนรักกาแฟ, ของที่ระลึกงานแต่ง, ของขวัญผู้ใหญ่\nStyle: คราฟท์, อบอุ่น",
    "Product: กระเป๋าผ้าแคนวาสปักลายใบไม้\nDescription: กระเป๋าผ้าใบหนา ทนทาน ปักลายด้วยมือ ดีไซน์รักษ์โลก\nSuitable for: ของแจกพนักงาน, ของที่ระลึก, ใช้ไปเรียน\nStyle: ธรรมชาติ, เรียบง่าย",
    "Product: นาฬิกาตั้งโต๊ะดิจิทัลลายไม้\nDescription: นาฬิกาบอกเวลาและอุณหภูมิ หน้าจอ LED ซ่อนใต้ผิวไม้\nSuitable for: ตกแต่งออฟฟิศ, ของขวัญปีใหม่\nStyle: โมเดิร์น, มินิมอล",
    "Product: ชุดก้านไม้หอมกระจายกลิ่น\nDescription: Reed Diffuser กลิ่นโอเชี่ยนเฟรช สดชื่นยาวนาน 30 วัน\nSuitable for: ของขวัญงานแต่ง, ของแจกพนักงาน\nStyle: สดชื่น, เรียบหรู",
    "Product: สมุดแพลนเนอร์ปกผ้าลินิน\nDescription: สมุดจดบันทึกรายปี กระดาษถนอมสายตา ปกหุ้มผ้าลินินสีครีม\nSuitable for: คนทำงาน, นักเรียน, ของขวัญวันเกิด\nStyle: มูจิ, มินิมอล",
    "Product: กล่องดนตรีไม้ไขลาน\nDescription: กล่องดนตรีไม้แกะสลัก เพลงคลาสสิก เสียงใสไพเราะ\nSuitable for: ของขวัญวันครบรอบ, ของขวัญเด็ก\nStyle: วินเทจ, คลาสสิก",
    "Product: ผ้าพันคอผ้าไหมพิมพ์ลายไทย\nDescription: ผ้าไหมเนื้อละเอียด พิมพ์ลายไทยประยุกต์ สีสันสดใส\nSuitable for: ของขวัญชาวต่างชาติ, ของขวัญผู้ใหญ่\nStyle: ไทยประยุกต์, หรูหรา",
    "Product: ชุดเครื่องเขียนโลหะสีทอง\nDescription: เซตปากกาและคลิปหนีบกระดาษ สีทองหรูหรา บรรจุในกล่องสวยงาม\nSuitable for: ของขวัญเลื่อนตำแหน่ง, ของขวัญผู้บริหาร\nStyle: ลักชูรี, ทางการ",
    "Product: กระเป๋าสตางค์หนังวัวแท้แบบพับ\nDescription: หนังแท้สัมผัสนุ่ม มีช่องใส่บัตร 8 ช่อง พร้อมช่องซิปใส่เหรียญ\nSuitable for: ของขวัญวันเกิด, ของขวัญผู้ชาย, งานเกษียณ\nStyle: เรียบหรู, คลาสสิก",
    "Product: ชุดชงชาเซรามิกสไตล์ญี่ปุ่น\nDescription: กาน้ำชาพร้อมถ้วย 4 ใบ ลายคลื่นทะเลญี่ปุ่น บรรจุกล่องไม้\nSuitable for: ของขวัญผู้ใหญ่, คนชอบดื่มชา, ของที่ระลึก\nStyle: เซน, ญี่ปุ่น",
    "Product: หมอนอิงกำมะหยี่สีเอิร์ธโทน\nDescription: หมอนอิงนุ่มพิเศษ ปลอกถอดซักได้ ขนาด 45x45 ซม.\nSuitable for: ตกแต่งโซฟา, ของขวัญขึ้นบ้านใหม่\nStyle: โมเดิร์น, อบอุ่น",
    "Product: แผ่นรองเม้าส์หนังสังเคราะห์ขนาดใหญ่\nDescription: แผ่นรองแบบ Desk Mat กันน้ำ ผิวสัมผัสเรียบหรู กว้าง 80 ซม.\nSuitable for: จัดโต๊ะคอม, ของขวัญพนักงานออฟฟิศ\nStyle: มินิมอล, มืออาชีพ",
    "Product: ร่มพับพกพาเคลือบ UV\nDescription: ร่มน้ำหนักเบา กันแดดและกันฝน แข็งแรงทนทานต่อลมแรง\nSuitable for: ของแจกอีเวนต์, ของขวัญพนักงาน\nStyle: ทันสมัย",
    "Product: ชุดปลูกแคคตัส DIY\nDescription: ในชุดประกอบด้วยกระถาง ดิน เมล็ดพันธุ์ และคู่มือการปลูก\nSuitable for: ของขวัญเด็ก, กิจกรรมยามว่าง, คนรักต้นไม้\nStyle: ธรรมชาติ, น่ารัก",
    "Product: ลำโพงไม้บลูทูธพกพา\nDescription: ลำโพงไร้สายดีไซน์ตัวเรือนไม้ ให้เสียงโทนอบอุ่น แบตเตอรี่อึด\nSuitable for: ของขวัญวันเกิด, ตกแต่งโต๊ะทำงาน\nStyle: วินเทจ, ธรรมชาติ",
    "Product: ผ้ากันเปื้อนผ้าลินินสไตล์คาเฟ่\nDescription: ผ้ากันเปื้อนแบบสายไขว้หลัง มีกระเป๋าหน้าใบใหญ่ เนื้อผ้าเกรดเอ\nSuitable for: คนชอบทำอาหาร, เจ้าของร้านกาแฟ, ของขวัญวันแม่\nStyle: มินิมอล, คาเฟ่",
    "Product: หูฟังครอบหูแบบตัดเสียงรบกวน\nDescription: หูฟังไร้สายระบบ ANC เบสแน่น ใส่สบายไม่บีบหู\nSuitable for: คนรักเสียงเพลง, ของขวัญรับปริญญา, คนเดินทางบ่อย\nStyle: เทค, ทันสมัย",
    "Product: ป้ายชื่อไม้สลักเลเซอร์\nDescription: ป้ายชื่อตั้งโต๊ะทำจากไม้สนแท้ สลักชื่อและตำแหน่งด้วยเลเซอร์ความละเอียดสูง\nSuitable for: ของขวัญเลื่อนตำแหน่ง, ของแจกพนักงาน\nStyle: ทางการ, อบอุ่น"
]

In [7]:
metadatas = [
    {"product_id": 1,"category": "stationery","event": "birthday,wedding,corporate","price": 189,"currency": "THB","seller_id": "seller_003","seller_name": "Retro Craft TH","stock": 84},
    {"product_id": 2,"category": "stationery","event": "retirement","price": 890,"currency": "THB","seller_id": "seller_004","seller_name": "WoodCraft","stock": 25},
    {"product_id": 3, "category": "home_decor", "event": "birthday,housewarming", "price": 350, "currency": "THB", "seller_id": "seller_005", "seller_name": "Scent & Soul", "stock": 45},
    {"product_id": 4, "category": "home_decor", "event": "graduation,reading", "price": 790, "currency": "THB", "seller_id": "seller_006", "seller_name": "Light Design", "stock": 20},
    {"product_id": 5, "category": "kitchenware", "event": "wedding,coffee_lover", "price": 420, "currency": "THB", "seller_id": "seller_007", "seller_name": "Ceramic Studio", "stock": 15},
    {"product_id": 6, "category": "fashion", "event": "corporate,souvenir", "price": 250, "currency": "THB", "seller_id": "seller_008", "seller_name": "Green Bag TH", "stock": 100},
    {"product_id": 7, "category": "gadget", "event": "new_year,office", "price": 550, "currency": "THB", "seller_id": "seller_009", "seller_name": "Woody Tech", "stock": 30},
    {"product_id": 8, "category": "lifestyle", "event": "wedding,corporate", "price": 390, "currency": "THB", "seller_id": "seller_010", "seller_name": "Aroma Fresh", "stock": 60},
    {"product_id": 9, "category": "stationery", "event": "birthday,student", "price": 290, "currency": "THB", "seller_id": "seller_011", "seller_name": "Minimal Note", "stock": 85},
    {"product_id": 10, "category": "gift", "event": "anniversary,children", "price": 1200, "currency": "THB", "seller_id": "seller_012", "seller_name": "Music Box Shop", "stock": 12},
    {"product_id": 11, "category": "fashion", "event": "foreigner,elderly", "price": 1500, "currency": "THB", "seller_id": "seller_013", "seller_name": "Thai Silk Co.", "stock": 25},
    {"product_id": 12, "category": "stationery", "event": "promotion,executive", "price": 2100, "currency": "THB", "seller_id": "seller_014", "seller_name": "Elite Office", "stock": 8},
    {"product_id": 13, "category": "fashion", "event": "birthday,retirement", "price": 1250, "currency": "THB", "seller_id": "seller_015", "seller_name": "Leather Master", "stock": 40},
    {"product_id": 14, "category": "kitchenware", "event": "elderly,souvenir", "price": 980, "currency": "THB", "seller_id": "seller_016", "seller_name": "Zen Ceramic", "stock": 18},
    {"product_id": 15, "category": "home_decor", "event": "housewarming", "price": 350, "currency": "THB", "seller_id": "seller_017", "seller_name": "Soft Living", "stock": 55},
    {"product_id": 16, "category": "office_supplies", "event": "corporate,office_setup", "price": 490, "currency": "THB", "seller_id": "seller_018", "seller_name": "Desk Decor", "stock": 120},
    {"product_id": 17, "category": "lifestyle", "event": "corporate,giveaway", "price": 290, "currency": "THB", "seller_id": "seller_019", "seller_name": "Everyday Carry", "stock": 200},
    {"product_id": 18, "category": "gardening", "event": "hobby,children", "price": 199, "currency": "THB", "seller_id": "seller_020", "seller_name": "Green Thumb", "stock": 65},
    {"product_id": 19, "category": "electronics", "event": "birthday,office", "price": 1590, "currency": "THB", "seller_id": "seller_021", "seller_name": "Wooden Sound", "stock": 22},
    {"product_id": 20, "category": "fashion", "event": "cooking,mother_day", "price": 450, "currency": "THB", "seller_id": "seller_022", "seller_name": "Cafe Wear", "stock": 38},
    {"product_id": 21, "category": "electronics", "event": "graduation,travel", "price": 3200, "currency": "THB", "seller_id": "seller_023", "seller_name": "Audio Tech", "stock": 15},
    {"product_id": 22, "category": "office_supplies", "event": "promotion,corporate", "price": 590, "currency": "THB", "seller_id": "seller_024", "seller_name": "Craft Sign", "stock": 50}
]

In [8]:
metadatas[0]

{'product_id': 1,
 'category': 'stationery',
 'event': 'birthday,wedding,corporate',
 'price': 189,
 'currency': 'THB',
 'seller_id': 'seller_003',
 'seller_name': 'Retro Craft TH',
 'stock': 84}

## Creating the embedding database with ChromaDB

In [9]:
from google.genai import types

class GeminiEmbeddingFunction(EmbeddingFunction):
  def __call__(self, input: Documents) -> Embeddings:
    EMBEDDING_MODEL_ID = "gemini-embedding-001"
    title = "Custom query"
    response = client.models.embed_content(
        model=EMBEDDING_MODEL_ID,
        contents=input,
        config=types.EmbedContentConfig(
          task_type="retrieval_document",
          title=title
        )
    )

    return [e.values for e in response.embeddings]

In [10]:
def create_chroma_db(records, name, metadata):
    chroma_client = chromadb.CloudClient(
        api_key=chroma_api_key,
        tenant='f69a80c1-ef1c-467e-be0a-4295c55b5ffb',
        database='testing'                              
    )
    db = chroma_client.get_or_create_collection(
        name=name,
        embedding_function=GeminiEmbeddingFunction()
    )
    db.add(
        documents=records,
        metadatas=metadata,
        ids=[str(i) for i in range(len(records))]
    )
    
    return db

In [11]:
db = create_chroma_db(records, "google_collection", metadatas)

C:\Users\Thunder\AppData\Local\Temp\ipykernel_25344\3949515305.py:9: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embedding_function=GeminiEmbeddingFunction()


In [38]:
db

Collection(name=google_collection)

In [45]:
sample_data = db.get(include=['documents', 'metadatas', 'embeddings'])

df = pd.DataFrame({
    "IDs": sample_data['ids'][:3],
    "Documents": sample_data['documents'][:3],
    "Metadata": [str(metadata)[:50] + "..." for metadata in sample_data['metadatas'][:3]],
    "Embeddings": [str(emb)[:50] + "..." for emb in sample_data['embeddings'][:3]]
})

df

,IDs,Documents,Metadata,Embeddings
0,0,Product: สมุดโน้ตกระดาษคราฟท์ปกแข็งลายวินเทจ\n...,"{'category': 'stationery', 'price': 189, 'even...",[-0.01804495 -0.02576489 -0.00643981 ... -0.00...
1,1,Product: ปากกาหมึกซึมลายไม้\nDescription: ปากก...,"{'currency': 'THB', 'seller_name': 'WoodCraft'...",[-0.01317799 -0.01078116 -0.00588682 ... 0.01...
2,2,Product: เทียนหอมไขถั่วเหลืองกลิ่นลาเวนเดอร์\n...,"{'seller_name': 'Scent & Soul', 'category': 'h...",[-0.01124499 -0.00911894 -0.01226668 ... 0.00...


In [57]:
def get_relevant_products(query, db, n_results, threshold):
    # 1. Query ข้อมูลพร้อมค่า distances
    results = db.query(
        query_texts=[query], 
        n_results=n_results,
        include=['documents', 'distances', 'metadatas'] # ดึง metadata มาด้วยถ้ามี
    )

    found_products = []
    
    # 2. วนลูปเช็คผลลัพธ์แต่ละตัวที่ได้มา
    # results['documents'][0] จะเป็น List ของสินค้า n ตัว
    for i in range(len(results['documents'][0])):
        distance = results['distances'][0][i]
        content = results['documents'][0][i]
        content_inspect = results['metadatas'][0][i]
        # print(content)
        
        # 3. ตรวจสอบว่าผ่านเกณฑ์ความเกี่ยวข้องไหม
        if distance <= threshold:
            content_results = f"Distance: {distance}, {content}, {content_inspect}"
            found_products.append(content_results)

    # 4. ตัดสินใจแสดงผล
    if not found_products:
        return "ไม่พบสินค้าที่เกี่ยวข้อง"
    
    # ส่งคืนเป็นรายการสินค้า (แยกบรรทัดเพื่อให้ไม่อ่านยาก)
    return "\n\n".join(found_products)

Query Test

ทดสอบการค้นหาข้อมูลจาก Vector Database(ยิ่ง Distance ใกล้ 0 ยิ่งแม่นมากๆ)

In [59]:
relevant_products = get_relevant_products(
    "ของขวัญให้แฟน ขอแบบน่ารักๆ",
    db,
    n_results=5,
    threshold=0.8
)
Markdown(relevant_products)

Distance: 0.3739817, Product: ชุดปลูกแคคตัส DIY
Description: ในชุดประกอบด้วยกระถาง ดิน เมล็ดพันธุ์ และคู่มือการปลูก
Suitable for: ของขวัญเด็ก, กิจกรรมยามว่าง, คนรักต้นไม้
Style: ธรรมชาติ, น่ารัก, {'price': 199, 'category': 'gardening', 'event': 'hobby,children', 'currency': 'THB', 'seller_id': 'seller_020', 'stock': 65, 'product_id': 18, 'seller_name': 'Green Thumb'}

Distance: 0.37404037, Product: กล่องดนตรีไม้ไขลาน
Description: กล่องดนตรีไม้แกะสลัก เพลงคลาสสิก เสียงใสไพเราะ
Suitable for: ของขวัญวันครบรอบ, ของขวัญเด็ก
Style: วินเทจ, คลาสสิก, {'seller_id': 'seller_012', 'price': 1200, 'currency': 'THB', 'seller_name': 'Music Box Shop', 'product_id': 10, 'category': 'gift', 'stock': 12, 'event': 'anniversary,children'}

Distance: 0.37485927, Product: ผ้าพันคอผ้าไหมพิมพ์ลายไทย
Description: ผ้าไหมเนื้อละเอียด พิมพ์ลายไทยประยุกต์ สีสันสดใส
Suitable for: ของขวัญชาวต่างชาติ, ของขวัญผู้ใหญ่
Style: ไทยประยุกต์, หรูหรา, {'event': 'foreigner,elderly', 'product_id': 11, 'currency': 'THB', 'seller_id': 'seller_013', 'category': 'fashion', 'seller_name': 'Thai Silk Co.', 'price': 1500, 'stock': 25}

Distance: 0.3776062, Product: ชุดถ้วยกาแฟเซรามิกทำมือ
Description: ถ้วยกาแฟงานปั้นมือ เคลือบสีเอิร์ธโทน เอกลักษณ์ไม่ซ้ำกันในแต่ละใบ
Suitable for: คนรักกาแฟ, ของที่ระลึกงานแต่ง, ของขวัญผู้ใหญ่
Style: คราฟท์, อบอุ่น, {'currency': 'THB', 'seller_name': 'Ceramic Studio', 'price': 420, 'event': 'wedding,coffee_lover', 'stock': 15, 'seller_id': 'seller_007', 'product_id': 5, 'category': 'kitchenware'}

Distance: 0.3791213, Product: เทียนหอมไขถั่วเหลืองกลิ่นลาเวนเดอร์
Description: เทียนหอมธรรมชาติ ไร้เขม่าดำ ช่วยผ่อนคลายและหลับสบาย
Suitable for: ของขวัญวันเกิด, ของขวัญขึ้นบ้านใหม่, คนชอบแต่งบ้าน
Style: มินิมอล, อบอุ่น, {'category': 'home_decor', 'seller_name': 'Scent & Soul', 'currency': 'THB', 'event': 'birthday,housewarming', 'product_id': 3, 'stock': 45, 'price': 350, 'seller_id': 'seller_005'}

### AI Agent

In [54]:
def make_prompt(query, relevant_products):
  escaped = relevant_products.replace("'", "").replace('"', "").replace("\n", " ")
  prompt = ("""
    คุณคือผู้ช่วยจัดการสต็อกสินค้า หน้าที่ของคุณคือดึงข้อมูลจาก 'PRODUCTS IN STOCK' มาตอบคำถาม 'QUESTION'
    
    กฎการตอบ:
    1. ตอบเฉพาะข้อมูลสินค้าที่เกี่ยวข้องเท่านั้น ห้ามแต่งประโยคทักทายหรือเกริ่นนำ
    2. หากข้อมูลใน PRODUCTS IN STOCK ไม่เกี่ยวข้องกับคำถาม ให้ตอบว่า "ไม่พบสินค้าที่เกี่ยวข้อง"
    3. ตอบคำถามตามรูปแบบที่กำหนดเท่านั้น
    
    รูปแบบการตอบ:
    [ชื่อสินค้า/รหัส] | ราคา: [ระบุราคา] | สถานะ: [ระบุสถานะ]       

    QUESTION: '{query}'
    PRODUCTS IN STOCK: '{relevant_products}'

    ANSWER:
  """).format(query=query, relevant_products=escaped)

  return prompt

In [61]:
query = "อยากได้ของขวัญน่ารักๆให้แฟน"
relevant_products = get_relevant_products(
    query,
    db,
    n_results=5,
    threshold=0.8
)
prompt = make_prompt(query, relevant_products)
Markdown(prompt)


    คุณคือผู้ช่วยจัดการสต็อกสินค้า หน้าที่ของคุณคือดึงข้อมูลจาก 'PRODUCTS IN STOCK' มาตอบคำถาม 'QUESTION'

    กฎการตอบ:
    1. ตอบเฉพาะข้อมูลสินค้าที่เกี่ยวข้องเท่านั้น ห้ามแต่งประโยคทักทายหรือเกริ่นนำ
    2. หากข้อมูลใน PRODUCTS IN STOCK ไม่เกี่ยวข้องกับคำถาม ให้ตอบว่า "ไม่พบสินค้าที่เกี่ยวข้อง"
    3. ตอบคำถามตามรูปแบบที่กำหนดเท่านั้น

    รูปแบบการตอบ:
    [ชื่อสินค้า/รหัส] | ราคา: [ระบุราคา] | สถานะ: [ระบุสถานะ]       

    QUESTION: 'อยากได้ของขวัญน่ารักๆให้แฟน'
    PRODUCTS IN STOCK: 'Distance: 0.3768306, Product: เทียนหอมไขถั่วเหลืองกลิ่นลาเวนเดอร์ Description: เทียนหอมธรรมชาติ ไร้เขม่าดำ ช่วยผ่อนคลายและหลับสบาย Suitable for: ของขวัญวันเกิด, ของขวัญขึ้นบ้านใหม่, คนชอบแต่งบ้าน Style: มินิมอล, อบอุ่น, {seller_id: seller_005, stock: 45, category: home_decor, currency: THB, seller_name: Scent & Soul, product_id: 3, price: 350, event: birthday,housewarming}  Distance: 0.3797943, Product: ชุดปลูกแคคตัส DIY Description: ในชุดประกอบด้วยกระถาง ดิน เมล็ดพันธุ์ และคู่มือการปลูก Suitable for: ของขวัญเด็ก, กิจกรรมยามว่าง, คนรักต้นไม้ Style: ธรรมชาติ, น่ารัก, {seller_id: seller_020, event: hobby,children, currency: THB, price: 199, category: gardening, seller_name: Green Thumb, product_id: 18, stock: 65}  Distance: 0.38140512, Product: ชุดถ้วยกาแฟเซรามิกทำมือ Description: ถ้วยกาแฟงานปั้นมือ เคลือบสีเอิร์ธโทน เอกลักษณ์ไม่ซ้ำกันในแต่ละใบ Suitable for: คนรักกาแฟ, ของที่ระลึกงานแต่ง, ของขวัญผู้ใหญ่ Style: คราฟท์, อบอุ่น, {price: 420, seller_name: Ceramic Studio, product_id: 5, currency: THB, stock: 15, event: wedding,coffee_lover, seller_id: seller_007, category: kitchenware}  Distance: 0.38379246, Product: กล่องดนตรีไม้ไขลาน Description: กล่องดนตรีไม้แกะสลัก เพลงคลาสสิก เสียงใสไพเราะ Suitable for: ของขวัญวันครบรอบ, ของขวัญเด็ก Style: วินเทจ, คลาสสิก, {price: 1200, product_id: 10, seller_name: Music Box Shop, stock: 12, currency: THB, category: gift, seller_id: seller_012, event: anniversary,children}  Distance: 0.3846208, Product: ผ้าพันคอผ้าไหมพิมพ์ลายไทย Description: ผ้าไหมเนื้อละเอียด พิมพ์ลายไทยประยุกต์ สีสันสดใส Suitable for: ของขวัญชาวต่างชาติ, ของขวัญผู้ใหญ่ Style: ไทยประยุกต์, หรูหรา, {currency: THB, stock: 25, seller_name: Thai Silk Co., event: foreigner,elderly, seller_id: seller_013, category: fashion, price: 1500, product_id: 11}'

    ANSWER:
  

In [62]:
MODEL_ID = "gemini-3-flash-preview"
answer = client.models.generate_content(
    model = MODEL_ID,
    contents = prompt
)
Markdown(answer.text)

เทียนหอมไขถั่วเหลืองกลิ่นลาเวนเดอร์ | ราคา: 350 THB | สถานะ: มีสินค้า (คงเหลือ 45)
ชุดปลูกแคคตัส DIY | ราคา: 199 THB | สถานะ: มีสินค้า (คงเหลือ 65)
กล่องดนตรีไม้ไขลาน | ราคา: 1,200 THB | สถานะ: มีสินค้า (คงเหลือ 12)